# Body Controller Demos: how to run them

Three live MediaPipe demos in one web page (`index.html`):

| Tab | What it does |
|---|---|
| **Fingers** | Counts the fingers you hold up on one or two hands, thumbs included (up to 10) |
| **Face** | Filters (sunglasses, devil horns, crown, mustache, clown nose) plus a **mood detector**: smile, frown, surprise, blink, wink |
| **Full body** | Erases a person and turns them into an otter, robot, X-ray skeleton, neon light trails or an anime girl |

**Three ways to run it. Pick one:**

1. **From this notebook in Google Colab:** run Step 0, then Option 1. The demo opens in its own browser tab.
2. **From this notebook on your own computer** (Jupyter, VS Code): run Step 0, then Option 2.
3. **No code at all:** open the GitHub Pages link in Option 3, or double-click `index.html` in Chrome.

At the end, **Under the hood** runs the finger counter and the mood detector in Python, so you can see the same ideas as code.

*Everything runs on your device. Your camera video is never uploaded. Works best in Chrome or Edge, with any webcam, including USB cameras like the OBSBOT.*

## Step 0: get the demo page

Finds `index.html` next to this notebook, or downloads it from GitHub. If both fail in Colab, it asks you to upload the file.

👉 **Set `HTML_URL`** to your repo's raw file link (on GitHub, open `index.html` and click **Raw**).

In [ ]:
import os, sys, time, urllib.request

HTML_URL = "https://raw.githubusercontent.com/crqzyexprees/mediapipe-demo/main/index.html"  # <- change me
IN_COLAB = "google.colab" in sys.modules

def fetch(url, path, tries=3):
    for i in range(tries):
        try:
            urllib.request.urlretrieve(url, path)
            return True
        except Exception as e:
            last = e
            time.sleep(1 + i)
    print(f"Download failed: {last}")
    return False

def get_demo_page():
    if os.path.exists("index.html") and os.path.getsize("index.html") > 1000:
        print("Found index.html next to this notebook.")
        return
    if "USERNAME" not in HTML_URL and fetch(HTML_URL, "index.html"):
        print("Downloaded index.html from GitHub.")
        return
    if IN_COLAB:
        from google.colab import files
        print("Upload index.html:")
        uploaded = files.upload()
        name = next(iter(uploaded))
        with open("index.html", "wb") as f:
            f.write(uploaded[name])
        return
    raise FileNotFoundError("Put index.html next to this notebook, or set HTML_URL, then run this cell again.")

get_demo_page()
print(f"Demo page ready ({os.path.getsize('index.html') // 1024} KB).")

### Start a tiny web server

Both Option 1 and Option 2 use this. It serves the folder this notebook is in, quietly, on the first free port from 8000. Running it twice is safe.

In [ ]:
import functools, http.server, socketserver, threading

class _QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *args):
        pass
    def end_headers(self):
        self.send_header("Cache-Control", "no-store")   # always serve the latest index.html
        super().end_headers()

class _Server(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True

def start_server():
    global server, PORT
    if globals().get("server"):
        return PORT
    handler = functools.partial(_QuietHandler, directory=os.getcwd())
    for port in range(8000, 8021):
        try:
            server = _Server(("127.0.0.1", port), handler)
            break
        except OSError:
            continue
    else:
        raise RuntimeError("Ports 8000-8020 are all busy.")
    PORT = port
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return PORT

PORT = start_server()
print(f"Serving on port {PORT}.")

## Option 1: Google Colab

Click the link this cell prints. The demo opens in **its own browser tab** through Colab’s secure proxy, so the camera works at full speed. Press **Start camera** and allow access.

- Switch demos with the tabs or keys **1**, **2**, **3**. Press **O** to cycle filters or characters.
- Using a USB webcam? Pick it from the **Camera** list that appears after you start.
- Full body: step out of frame, press **Capture empty background**, then walk back in.

Keep this notebook open while you use the demo, since the server runs inside it.

In [ ]:
if not IN_COLAB:
    print("Not in Colab: use Option 2 below.")
else:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, path="/index.html")

**Prefer it inside the notebook?** This shows the same page in the cell below. Some browsers block the camera inside notebook outputs. If it stays on “Press Start camera” or shows a permission error, use the link above.

In [ ]:
if IN_COLAB:
    from google.colab import output
    try:
        output.serve_kernel_port_as_iframe(PORT, path="/index.html", height=900)
    except TypeError:   # older Colab versions take fewer options
        output.serve_kernel_port_as_iframe(PORT)

## Option 2: your own computer

Opens the demo in your default browser from the local server. In Chrome or Edge you can also just **double-click `index.html`**: no server needed.

In [ ]:
import webbrowser

if IN_COLAB:
    print("You're in Colab: use Option 1 above.")
else:
    url = f"http://localhost:{PORT}/index.html#fingers"
    print(f"Demo running at {url}")
    print("Direct links: #fingers  #face-horns  #face-none (mood only)  #body-otter  #body-robot  #body-skeleton  #body-neon  #body-anime")
    webbrowser.open(url)

When you're done, stop the server:

In [ ]:
if globals().get("server"):
    server.shutdown(); server.server_close(); server = None
    print("Server stopped.")
else:
    print("No server is running.")

## Option 3: no code at all

Share the GitHub Pages link. It works on any device with a camera, phones included:

**https://crqzyexprees.github.io/mediapipe-demo/**

Jump straight to a demo: [`#fingers`](https://crqzyexprees.github.io/mediapipe-demo/#fingers) · [`#face-horns`](https://crqzyexprees.github.io/mediapipe-demo/#face-horns) · [`#body-otter`](https://crqzyexprees.github.io/mediapipe-demo/#body-otter) · [`#body-robot`](https://crqzyexprees.github.io/mediapipe-demo/#body-robot)

Set it up once: upload `index.html` to a public repo, go to **Settings → Pages**, pick the `main` branch and save.

Want other models? [MediaPipe Studio](https://mediapipe-studio.webapps.google.com/home) has live demos of every task.

### Using an OBSBOT webcam

The page detects OBSBOT cameras by name and shows extra controls:

- **Zoom slider and Re-center**, when the browser exposes them. Press **Enable zoom and gimbal controls** if they don’t appear. Chrome then asks for permission to move the camera.
- **Full body** zooms out automatically so the whole body fits.
- **Turn off AI tracking** before capturing the background, since the trick needs a still camera. If the camera moves anyway, the page notices and asks you to capture the background again.
- The OBSBOT responds to some hand gestures itself, which can move or zoom it during the Fingers demo. If your model allows it, turn gesture control off in **OBSBOT Center**.

---
## Under the hood: the same ideas in Python

The web page runs MediaPipe in JavaScript, but the models and the logic are identical in Python.

In [ ]:
# Install MediaPipe (about 30 seconds in Colab). If Colab asks you to restart, restart and run Step 0 again.
%pip install -q mediapipe==1.0.1

Take a photo with your webcam (Colab only), or the cells use Google’s sample photos. Set `USE_WEBCAM = False` to always use the samples.

In [ ]:
USE_WEBCAM = True

def take_photo(filename="photo.jpg"):
    """Shows the webcam with a 'Take photo' button (Colab only)."""
    from base64 import b64decode
    from IPython.display import Javascript, display
    from google.colab.output import eval_js
    display(Javascript('''
      async function takePhoto() {
        const box = document.createElement("div");
        const video = document.createElement("video");
        const button = document.createElement("button");
        button.textContent = "Take photo";
        video.style.cssText = "display:block;max-width:100%";
        box.append(video, button);
        document.body.appendChild(box);
        const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 1280, height: 720 } });
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => (button.onclick = resolve));
        const canvas = document.createElement("canvas");
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext("2d").drawImage(video, 0, 0);
        stream.getTracks().forEach((t) => t.stop());
        box.remove();
        return canvas.toDataURL("image/jpeg", 0.92);
      }'''))
    data = eval_js("takePhoto()")
    with open(filename, "wb") as f:
        f.write(b64decode(data.split(",")[1]))
    return filename

def get_photo(sample_url, sample_name):
    if USE_WEBCAM and IN_COLAB:
        try:
            return take_photo()
        except Exception as e:
            print(f"Webcam snapshot didn't work ({type(e).__name__}), using a sample photo.")
    if not os.path.exists(sample_name):
        fetch(sample_url, sample_name)
    return sample_name

def get_model(url):
    path = url.rsplit("/", 1)[-1]
    if not os.path.exists(path):
        fetch(url, path)
    return path

### Finger counter

- **Fingers:** a finger is *up* when its tip (8, 12, 16, 20) is farther from the wrist (0) than its middle joint (6, 10, 14, 18).
- **Thumb:** it folds sideways across the palm, so it’s measured from the pinky base (17) instead. It’s *out* when its tip (4) is farther from 17 than its knuckle (3) is.

In [ ]:
import math
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

hand_model = get_model("https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task")
hands = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=hand_model), num_hands=2))

photo = get_photo("https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg", "sample_hands.jpg")
result = hands.detect(mp.Image.create_from_file(photo))

def dist(a, b):
    return math.hypot(a.x - b.x, a.y - b.y)

def count_fingers(hand):
    up = sum(dist(hand[t], hand[0]) > dist(hand[j], hand[0]) for t, j in zip([8, 12, 16, 20], [6, 10, 14, 18]))
    thumb_out = dist(hand[4], hand[17]) > dist(hand[3], hand[17]) * 1.05   # 5% margin so it doesn't flicker
    return up + thumb_out

frame = cv2.cvtColor(cv2.imread(photo), cv2.COLOR_BGR2RGB)
counts = []
for hand in result.hand_landmarks:
    counts.append(count_fingers(hand))
    vision.drawing_utils.draw_landmarks(frame, hand, vision.HandLandmarksConnections.HAND_CONNECTIONS)

title = (" + ".join(map(str, counts)) + (f" = {sum(counts)}" if len(counts) > 1 else "")) if counts else "No hands found"
plt.figure(figsize=(7, 7)); plt.imshow(frame); plt.axis("off"); plt.title(f"Fingers up: {title}", fontsize=16); plt.show()

### Mood detector

The Face Landmarker also returns 52 **blendshapes**: scores from 0 to 1 for things like `mouthSmileLeft`, `browDownLeft` or `jawOpen`. The page’s mood detector combines them like this:

In [ ]:
face_model = get_model("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task")
faces = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=face_model), output_face_blendshapes=True, num_faces=1))

photo = get_photo("https://storage.googleapis.com/mediapipe-assets/business-person.png", "sample_face.png")
result = faces.detect(mp.Image.create_from_file(photo))

def read_mood(cats):
    s = {c.category_name: c.score for c in cats}
    avg = lambda a, b: (s[a] + s[b]) / 2
    mood = {
        "smile": avg("mouthSmileLeft", "mouthSmileRight"),
        "frown": min(1, max(avg("mouthFrownLeft", "mouthFrownRight") * 1.8, avg("browDownLeft", "browDownRight") * 1.1)),
        "surprise": min(1, 0.55 * max(s["browInnerUp"], avg("browOuterUpLeft", "browOuterUpRight"))
                          + 0.45 * max(s["jawOpen"], avg("eyeWideLeft", "eyeWideRight"))),
        "mouth open": s["jawOpen"],
    }
    if mood["smile"] > 0.4:
        mood["frown"] *= 0.4   # a big smile pulls the brows down too
    return mood

if not result.face_blendshapes:
    print("No face found. Try another photo.")
else:
    mood = read_mood(result.face_blendshapes[0])
    labels = {"smile": ("Smiling", 0.45), "frown": ("Frowning", 0.42), "surprise": ("Surprised", 0.42)}
    hits = [(v, labels[k][0]) for k, v in mood.items() if k in labels and v > labels[k][1]]
    headline = max(hits)[1] if hits else "Neutral"
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [1.1, 1]})
    a1.imshow(cv2.cvtColor(cv2.imread(photo), cv2.COLOR_BGR2RGB)); a1.axis("off"); a1.set_title(headline, fontsize=18)
    a2.barh(list(mood)[::-1], list(mood.values())[::-1], color="#FF8A3D"); a2.set_xlim(0, 1); a2.set_title("Mood scores")
    plt.tight_layout(); plt.show()

### Try changing it

- Print every blendshape: `for c in result.face_blendshapes[0]: print(c.category_name, round(c.score, 2))`
- Make a **wink detector**: one of `eyeBlinkLeft` / `eyeBlinkRight` above 0.5 and the other below 0.2.
- Detect a **pinch** with the hand model: `dist(hand[4], hand[8]) < 0.05`.

In `index.html`, the same logic lives in `countHand()`, `readMood()`, and one drawing function per filter and character, like `drawHorns()`, `drawRobot()` or `drawAnime()`.

Docs: [MediaPipe guides](https://ai.google.dev/edge/mediapipe/solutions/guide) · Made by ml@psu